# RAG simple 100% local (Ollama + ChromaDB)

Version notebook de `rag_simple.py`.

**Entorno para correrlo:** kernel `.venv` (Python 3.11, raiz del repo). Ver la ultima celda del notebook ("A que entorno conectarse") para instrucciones.

**Requisitos previos:**
```
ollama pull nomic-embed-text   # modelo de embeddings
ollama pull gemma3             # modelo generador
```
y tener `ollama serve` corriendo (o el launcher `iniciar_rag_demo.ps1` lo levanta solo).

**Flujo:**
1. Indexar: cada documento se convierte en un vector (embedding) y se guarda en Chroma.
2. Recuperar: la pregunta del usuario tambien se convierte en vector y se buscan los documentos mas parecidos (similitud coseno) en Chroma.
3. Generar: se arma un prompt con la pregunta + los documentos recuperados como contexto, y se lo pasamos a Gemma para que responda basandose en ese contexto.

In [1]:
import sys

import chromadb
import ollama

EMBED_MODEL = "nomic-embed-text"
CHAT_MODEL = "gemma3"
COLLECTION_NAME = "clase_rag_demo"
TOP_K = 2

## Corpus de ejemplo (documentos "de la clase")

In [2]:
DOCUMENTOS = [
    {
        "id": "doc1",
        "text": (
            "RAG (Retrieval-Augmented Generation) es una tecnica que combina la "
            "busqueda de informacion con la generacion de texto: primero se recuperan "
            "documentos relevantes de una base de conocimiento, y luego un modelo de "
            "lenguaje genera la respuesta usando esos documentos como contexto."
        ),
    },
    {
        "id": "doc2",
        "text": (
            "Una base de datos vectorial almacena embeddings (vectores numericos que "
            "representan el significado de un texto) y permite buscar los vectores mas "
            "similares a una consulta usando metricas como la distancia coseno o euclidiana."
        ),
    },
    {
        "id": "doc3",
        "text": (
            "Ollama es una herramienta que permite correr modelos de lenguaje grandes "
            "de forma local, sin depender de una API en la nube. Soporta modelos como "
            "Gemma, Llama y modelos de embeddings como nomic-embed-text."
        ),
    },
    {
        "id": "doc4",
        "text": (
            "ChromaDB es una base de datos vectorial embebida, pensada para prototipos "
            "y aplicaciones RAG: se puede usar en memoria o persistir en disco, sin "
            "necesidad de levantar un servidor aparte."
        ),
    },
    {
        "id": "doc5",
        "text": (
            "El pipeline de un sistema RAG tipico tiene tres etapas: ingesta y chunking "
            "de documentos, indexado en una base vectorial, y en tiempo de consulta, "
            "recuperacion de los chunks mas relevantes seguida de generacion de la respuesta."
        ),
    },
]

## Funciones del pipeline

In [3]:
def build_index() -> chromadb.Collection:
    """Crea (o recrea) la coleccion de Chroma y la llena con los embeddings de DOCUMENTOS."""
    client = chromadb.PersistentClient(path="./chroma_db")  # persiste en disco, no solo en memoria
    # Si ya existe de una corrida anterior, la recreamos para que quede limpia.
    if COLLECTION_NAME in [c.name for c in client.list_collections()]:
        client.delete_collection(COLLECTION_NAME)
    collection = client.create_collection(COLLECTION_NAME)

    for doc in DOCUMENTOS:
        # Cada documento se convierte en un vector numerico (embedding) via Ollama...
        embedding = ollama.embeddings(model=EMBED_MODEL, prompt=doc["text"])["embedding"]
        # ...y se guarda en Chroma junto con su texto original y un id unico.
        collection.add(
            ids=[doc["id"]],
            embeddings=[embedding],
            documents=[doc["text"]],
        )
    return collection


def retrieve(collection: chromadb.Collection, pregunta: str, k: int = TOP_K) -> list[str]:
    """Devuelve los k documentos mas parecidos (por embedding) a la pregunta."""
    query_embedding = ollama.embeddings(model=EMBED_MODEL, prompt=pregunta)["embedding"]
    resultados = collection.query(query_embeddings=[query_embedding], n_results=k)
    return resultados["documents"][0]  # [0] porque query() soporta multiples consultas a la vez


def generar_respuesta(pregunta: str, contexto: list[str]) -> str:
    """Arma un prompt con el contexto recuperado y le pide a Gemma que responda solo con eso."""
    contexto_str = "\n\n".join(f"- {c}" for c in contexto)
    prompt = f"""Respondé la pregunta usando SOLO la informacion del contexto. Si el contexto no alcanza, decilo.

Contexto:
{contexto_str}

Pregunta: {pregunta}

Respuesta:"""

    response = ollama.chat(model=CHAT_MODEL, messages=[{"role": "user", "content": prompt}])
    return response["message"]["content"]


def rag(collection: chromadb.Collection, pregunta: str) -> None:
    """Orquesta el flujo completo: recuperar contexto y generar la respuesta, con logs por consola."""
    print(f"\n=== Pregunta: {pregunta} ===")
    contexto = retrieve(collection, pregunta)
    print("\n--- Documentos recuperados ---")
    for c in contexto:
        print(f"  * {c[:90]}...")
    respuesta = generar_respuesta(pregunta, contexto)
    print("\n--- Respuesta del modelo ---")
    print(respuesta)

## Indexar documentos

In [4]:
print("Indexando documentos en ChromaDB...")
collection = build_index()
print(f"Listo: {collection.count()} documentos indexados.")

Indexando documentos en ChromaDB...
Listo: 5 documentos indexados.


## Probar con una pregunta

Cambia el texto de `pregunta` y volve a correr la celda para probar distintas consultas.

In [5]:
pregunta = "¿Qué es RAG?"
rag(collection, pregunta)


=== Pregunta: ¿Qué es RAG? ===

--- Documentos recuperados ---
  * RAG (Retrieval-Augmented Generation) es una tecnica que combina la busqueda de informacion...
  * El pipeline de un sistema RAG tipico tiene tres etapas: ingesta y chunking de documentos, ...

--- Respuesta del modelo ---
RAG (Retrieval-Augmented Generation) es una técnica que combina la búsqueda de información con la generación de texto.



## (Opcional) Modo interactivo por consola

Igual que correr `python rag_simple.py` sin argumentos: pide preguntas por `input()` hasta que escribas `salir`.

In [6]:
modo_interactivo = False  # poner en True para activar el loop

if modo_interactivo:
    print("Escribí una pregunta (o 'salir' para terminar):")
    while True:
        pregunta = input("\n> ").strip()
        if pregunta.lower() in {"salir", "exit", "quit"}:
            break
        if pregunta:
            rag(collection, pregunta)

## A qué entorno conectarse

Este proyecto ya tiene un venv listo en la raíz del repo: `.venv` (Python 3.11.9), con `ollama`, `chromadb` e `ipykernel` instalados.

**En VS Code / Cursor:** arriba a la derecha del notebook, click en "Select Kernel" → "Python Environments" → elegí el que apunte a:
```
c:\Users\guill\OneDrive\Documentos\llm\.venv\Scripts\python.exe
```

**Desde Jupyter (navegador):** activá el venv y abrí jupyter desde ahí:
```powershell
cd c:\Users\guill\OneDrive\Documentos\llm
.\.venv\Scripts\Activate.ps1
jupyter notebook RAG\rag_simple.ipynb
```

**Antes de correrlo**, asegurate de tener Ollama corriendo y los modelos descargados:
```powershell
ollama serve            # si no esta corriendo ya
ollama pull nomic-embed-text
ollama pull gemma3
```
(el launcher `RAG\iniciar_rag_demo.ps1` hace estos chequeos automáticamente para la version .py).